# C04. Групповая утечка: проверка процедуры
Синтетические кадры 30 роликов, 12 кадров на ролик. Признаки — числовой сигнал, смещение источника и шум; цель бинарная. Финальный test фиксирован заранее: ролики 24…29. Подбор ведётся только на оставшихся роликах.

Сравним validation по кадрам и по роликам. Высокая оценка сама по себе не доказывает корректность. Запуск: зависимости из `requirements-projects.txt`, затем Restart + Run All. Полноценный PyTorch не требуется.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
rng=np.random.default_rng(7)
groups=np.repeat(np.arange(30),12)
source_bias=rng.normal(size=30)
signal=rng.normal(size=len(groups))
X=np.column_stack([signal,source_bias[groups],rng.normal(size=len(groups))])
y=(signal+source_bias[groups]+rng.normal(scale=.5,size=len(groups))>0).astype(int)
final_mask=groups>=24
X_dev,y_dev,g_dev=X[~final_mask],y[~final_mask],groups[~final_mask]
X_test,y_test,g_test=X[final_mask],y[final_mask],groups[final_mask]
assert not set(g_dev)&set(g_test)
assert X_dev.shape[1]==3

In [ ]:
frame_train,frame_valid=train_test_split(np.arange(len(y_dev)),test_size=.25,random_state=13,stratify=y_dev)
group_train,group_valid=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=13).split(X_dev,y_dev,g_dev))
assert not set(g_dev[group_train])&set(g_dev[group_valid])
assert not set(g_dev[group_valid])&set(g_test)
print('Общие ролики при кадровом split:',sorted(set(g_dev[frame_train])&set(g_dev[frame_valid])))
print('Общие ролики при групповом split:',sorted(set(g_dev[group_train])&set(g_dev[group_valid])))

In [ ]:
def metrics(y,pred):
    return dict(accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0))
# В этом notebook precision/recall при нулевом знаменателе = 0 (явное соглашение).
results=[]
for split,(train,valid) in {'frames':(frame_train,frame_valid),'groups':(group_train,group_valid)}.items():
    baseline=DummyClassifier(strategy='most_frequent').fit(X_dev[train],y_dev[train])
    results.append(dict(split=split,model='baseline',**metrics(y_dev[valid],baseline.predict(X_dev[valid]))))
    for c in [.1,1.,10.]:
        model=make_pipeline(StandardScaler(),LogisticRegression(C=c,random_state=0,max_iter=1000))
        model.fit(X_dev[train],y_dev[train])
        # scaler обучен только по train этого разбиения.
        np.testing.assert_allclose(model[0].mean_,X_dev[train].mean(axis=0))
        results.append(dict(split=split,model=f'C={c}',C=c,**metrics(y_dev[valid],model.predict(X_dev[valid]))))
table=pd.DataFrame(results)
table

In [ ]:
# Выбираем C по recall группового validation; равенство решает меньший C.
candidates=table[(table.split=='groups')&table.C.notna()]
best=candidates.sort_values(['recall','C'],ascending=[False,True]).iloc[0]
chosen=float(best.C)
final=make_pipeline(StandardScaler(),LogisticRegression(C=chosen,random_state=0,max_iter=1000)).fit(X_dev,y_dev)
final_baseline=DummyClassifier(strategy='most_frequent').fit(X_dev,y_dev)
final_table=pd.DataFrame([
    dict(model='baseline',**metrics(y_test,final_baseline.predict(X_test))),
    dict(model=f'Pipeline C={chosen}',**metrics(y_test,final.predict(X_test)))
])
final_table

## Разбор и самостоятельная часть
Параметр выбран по групповому validation, затем модель обучена на всей development-части. Финальный test использован только после выбора. Повторный подбор по финальной таблице нарушил бы этот порядок.

1. Объясните разницу между общими роликами и общими кадрами.
2. Измените random_state разбиения development-части, сохранив финальные ролики. Оцените устойчивость выбора без подбора под test.
3. Укажите, какие признаки были бы доступны в момент реального предсказания. Синтетический смещённый признак не является обоснованным реальным датчиком.
4. Напишите вывод с числами из таблицы, но не обещайте, что групповой split обязательно даёт меньшую оценку. Здесь проверяется отсутствие утечки и воспроизводимость процедуры.

Источник: https://scikit-learn.org/stable/common_pitfalls.html